In [47]:
import pandas as pd

In [48]:
import sqlalchemy

In [49]:
dados = {
    "Competição": [
        "Premier League", 
        "FA Cup", 
        "League Cup", 
        "Champions League", 
        "Europa League", 
        "FIFA Club World Cup", 
        "Copa Itália"
    ],
    "Manchester United": [20, 12, 5, 3, 1, 1, None],  # Exemplo com números fictícios
    "Liverpool": [19, 8, 9, 6, 0, 1, None]  # Exemplo com números fictícios
}

In [50]:
df = pd.DataFrame(dados)

In [51]:
df

,Competição,Manchester United,Liverpool
0,Premier League,20.0,19.0
1,FA Cup,12.0,8.0
2,League Cup,5.0,9.0
3,Champions League,3.0,6.0
4,Europa League,1.0,0.0
5,FIFA Club World Cup,1.0,1.0
6,Copa Itália,NaN,NaN


## Excluir a linha da Copa Itália, onde ambos não disputam a competição:

In [52]:
from sqlalchemy import MetaData, Table, inspect, create_engine

In [53]:
engine = create_engine('sqlite:///:memory:')

In [54]:
#Enviando dados ao banco de dados local:
df.to_sql('united_x_liverpool', engine)

7

In [55]:
#Verificado tabelas salvas I:
inspector = inspect(engine)

In [56]:
#Verificado tabelas salvas II:
inspector.get_table_names()

['united_x_liverpool']

In [57]:
#Verificando a linha "Copa Itália":
query = 'SELECT * FROM united_x_liverpool WHERE Competição == "Copa Itália"'

In [58]:
pd.read_sql(query, engine)

,index,Competição,Manchester United,Liverpool
0,6,Copa Itália,None,None


In [59]:
#Deletando linha "Copa Itália":
query_deletar = 'DELETE FROM united_x_liverpool WHERE Competição == "Copa Itália"'

In [60]:
from sqlalchemy import text #Necessário para executar requisição

In [61]:
with engine.connect() as conn:
    conn.execute(text(query_deletar)) 
    conn.commit() #Fechando execução

In [62]:
pd.read_sql_table('united_x_liverpool', engine)

,index,Competição,Manchester United,Liverpool
0,0,Premier League,20.0,19.0
1,1,FA Cup,12.0,8.0
2,2,League Cup,5.0,9.0
3,3,Champions League,3.0,6.0
4,4,Europa League,1.0,0.0
5,5,FIFA Club World Cup,1.0,1.0


## Alterar a informação errada (por exemplo, número de Champions League do Liverpool, caso seja incorreto):

In [66]:
query_update = 'UPDATE united_x_liverpool SET "Manchester United" = 2.0 WHERE Competição == "FIFA Club World Cup"'
#Títulos do Manchester United estão errados

In [67]:
with engine.connect() as conn:
    conn.execute(text(query_update))
    conn.commit()

In [68]:
pd.read_sql_table('united_x_liverpool', engine)

,index,Competição,Manchester United,Liverpool
0,0,Premier League,20.0,19.0
1,1,FA Cup,12.0,8.0
2,2,League Cup,5.0,9.0
3,3,Champions League,3.0,6.0
4,4,Europa League,1.0,0.0
5,5,FIFA Club World Cup,2.0,1.0
